# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze data from the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All data schema entities (record sets, fields, columns) are referenced by their `@id` for clarity and reproducibility.

### Dataset Source
The dataset is provided as a Croissant schema at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant metadata and access a summary from the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review the available record sets and their corresponding `@id`s and fields. This will help you identify which parts of the data to load for analysis.

In [ ]:
# List all record sets with their `@id` and field structure
recordsets = dataset.record_sets

if not recordsets:
    print("No record sets were found in this schema. Trying to discover from schema details ...")

    if hasattr(dataset.metadata, 'recordSet'):
        # Some exporters may not map 'RecordSet' into `record_sets`, but only in `metadata.recordSet`
        rs = dataset.metadata.recordSet
        if isinstance(rs, list):
            recordsets = rs
        else:
            recordsets = [rs]
    else:
        print("Could not find record sets.")

for rs in recordsets:
    print(f"Record set name: {getattr(rs, 'name', '(no name)')} | @id: {getattr(rs, '@id', '(no id)')}")
    print("  Fields and columns with @id:")
    for field in getattr(rs, 'field', []):
        print(f"   - Field: {getattr(field, 'name', '(no name)')} (@id: {getattr(field, '@id', '(no id)')}) | dataType: {getattr(field, 'dataType', '(no data type)')}")
        if hasattr(field, 'column'):
            for col in getattr(field, 'column', []):
                print(f"     Column: {getattr(col, 'name', '(no col name)')} (@id: {getattr(col, '@id', '(no id)')})")

## 3. Data Extraction
Load data from the primary record set into a DataFrame. Make sure to use the record set `@id` identified in the overview above.

In [ ]:
# Collect all available record set `@id`s
record_set_ids = []
for rs in dataset.record_sets:
    rs_id = getattr(rs, '@id', None)
    if rs_id:
        record_set_ids.append(rs_id)

# Fallback: If recordSets is empty, check metadata
if not record_set_ids and hasattr(dataset.metadata, 'recordSet'):
    rs_list = dataset.metadata.recordSet
    if isinstance(rs_list, list):
        for rs in rs_list:
            if isinstance(rs, dict) and '@id' in rs:
                record_set_ids.append(rs['@id'])
            elif hasattr(rs, '@id'):
                record_set_ids.append(rs.@id)
    elif isinstance(rs_list, dict) and '@id' in rs_list:
        record_set_ids.append(rs_list['@id'])

# Show discovered record sets
print(f"Record sets available: {record_set_ids}")

# If no record sets found, stop
if not record_set_ids:
    raise ValueError("No record sets found in schema.")

# Load all record sets into DataFrames using their @id
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading data from record set {rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[rs_id])} records, columns: {dataframes[rs_id].columns.tolist()}")
    else:
        print(f"No records found for {rs_id}")

# Pick the main record set (first one for demonstration)
main_record_set_id = record_set_ids[0]
print(f'Using record set @id = {main_record_set_id}')
df = dataframes[main_record_set_id]
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's process a numeric field (e.g., Coverage, MW) from the dataset. We reference the field by its `@id`. Typical EDA steps include filtering records by threshold, normalization, and grouping by a categorical variable.

In [ ]:
import numpy as np

# Try to identify a numeric field
numeric_field_candidates = [col for col in df.columns if any(x in col.lower() for x in ['coverage', 'mw', 'abundance', 'count', 'peptide', 'intensity', 'quantity'])]
print(f"Numeric field candidates: {numeric_field_candidates}")

# Select a numeric field. If available, e.g., 'MW', otherwise take the first candidate.
numeric_field = None
for key in ['MW', 'MolecularWeight', 'Abundance', 'coverage', 'Coverage', 'peptide_count', 'PeptideCount']:
    if key in df.columns:
        numeric_field = key
        break
if not numeric_field and numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
if not numeric_field:
    raise ValueError("No numeric field found in data.")

print(f"Selected numeric field: {numeric_field}")
# Attempt to convert the field to numeric, handling coercion
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Set a filtering threshold (e.g., MW > 15000)
threshold = df[numeric_field].quantile(0.75) if not df[numeric_field].isnull().all() else 0
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"After filtering: {len(filtered_df)} records where {numeric_field} > {threshold:.2f}")
display(filtered_df[[numeric_field]].head())

# Normalize selected numeric field for filtered records
norm_col = f"{numeric_field}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"First 5 normalized values:")
display(filtered_df[[numeric_field, norm_col]].head())

# Try grouping by a likely categorical column, e.g., 'description', or first non-numeric column
categorical_candidates = [c for c in df.columns if df[c].dtype == 'O' and c.lower() != 'id']
group_field = None
for g in ['description', 'Description', 'Protein', 'Gene', 'Accession', 'accession']:
    if g in df.columns:
        group_field = g
        break
if not group_field and categorical_candidates:
    group_field = categorical_candidates[0]

if group_field:
    grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean of {numeric_field} by {group_field} (showing up to 5 groups):")
    display(grouped.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized values, as well as the grouped mean by the categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Plot normalized values if available
if norm_col in filtered_df:
    plt.figure(figsize=(7, 4))
    sns.histplot(filtered_df[norm_col].dropna(), bins=30, kde=True)
    plt.title(f"Normalized {numeric_field} in Filtered Records")
    plt.xlabel(f"{numeric_field} (normalized)")
    plt.ylabel("Count")
    plt.show()

if group_field:
    # Bar plot for grouped means (top N categories for visualization)
    N = 10
    grouped_short = grouped.sort_values(numeric_field, ascending=False)[:N]
    plt.figure(figsize=(9,5))
    sns.barplot(x=numeric_field, y=group_field, data=grouped_short)
    plt.title(f"Top {N} groups by mean {numeric_field}")
    plt.show()

## 6. Conclusion
In this notebook, we accessed and explored the FAIR² mass spectrometry dataset using `mlcroissant` and referenced all entities by their Croissant `@id`. We loaded a record set, filtered and normalized a selected numeric field, grouped records by categorical attributes, and visualized key patterns in the data. 

Such approaches facilitate robust, reproducible FAIR data workflows, allowing for easy extension to more complex analyses and integration into machine learning pipelines.